# Ingestione sui tre database da CSV preparato
Carichiamo soltanto i primi **100.000 record** già preparati. Selezionare `postgres`, `mongo` oppure `cockroach` in YAML e avviare solo il DB scelto con Jupyter. Nessun download o ID rigenerato.
I tre adapter derivano il contratto da `src/schema.py`. SQL usa BIGINT/INTEGER/TEXT; Mongo usa solo `_id` BSON Int64, gli altri interi BSON Int32 e null espliciti. Il validatore strict richiede tutte le 32 colonne logiche e l'audit $type controlla i tipi realmente salvati. Nessun indice secondario viene creato durante l'ingestione.
Percorso manuale didattico riservato a 100K e a una destinazione verificata: non usarlo per ricaricare le campagne concluse. `ABILITA_INGESTIONE=False` impedisce qualsiasi lettura estesa del CSV, ingestione o reset anche con `Run All`. Le conferme nei moduli restano obbligatorie.


In [ ]:
from pathlib import Path
import json
import logging
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.config import load_config, selected_dataset
from src.databases.factory import create_adapter
from src.ingestion import ingest_csv, IngestionError
from src.query_validation import build_reference, verify_database, capture_database_snapshot, compare_postgres_snapshot, read_postgres_snapshot, read_mongo_snapshot, compare_reference_snapshots
from src.results import ResultRun, result_digest
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
config = load_config(ROOT / 'config' / 'benchmark.yaml')
print(json.dumps(config.manifest(), indent=2))
ABILITA_INGESTIONE = False
if ABILITA_INGESTIONE is True and config.dataset_size != 100000:
    raise ValueError('Percorso manuale limitato a 100.000 record: per altre dimensioni usare il runner con autorizzazione esplicita.')


## Verifica del checkpoint e riferimento indipendente
Verifichiamo i checksum del bundle preparato, selezioniamo il CSV da 100k e calcoliamo gli attesi: tutte le nullabilità, Q1 su ID diversi e riproducibili, conteggi JFK/range e medie per compagnia. Le medie ignorano solo None. Tutto questo è **fuori dal tempo di ingestione**.

In [ ]:
if ABILITA_INGESTIONE is True:
    csv_path, artifact = selected_dataset(config)
    reference = build_reference(csv_path, config.dataset_size, artifact['sha256'], config.q1_ids, config.parameters)
    print('CSV:', csv_path.name, '| record:', reference['dataset_size'], '| SHA-256:', reference['dataset_sha256'])
    print('Q2:', reference['Q2'], '| Q3:', reference['Q3'], '| gruppi Q4:', len(reference['Q4']))
    print('Sequenza Q1:', reference['q1_ids'])
    if config.database in ('mongo', 'cockroach'):
        read_postgres_snapshot(reference, config.postgres_snapshot)
        print('Riferimento PostgreSQL compatibile:', config.postgres_snapshot)
    if config.database == 'cockroach':
        read_mongo_snapshot(reference, config.mongo_snapshot)
        print('Riferimento Mongo compatibile:', config.mongo_snapshot)
else:
    print('Operazione disabilitata: ABILITA_INGESTIONE=False.')


## Conferma esplicita e confini della misura
Il default `ingestion.confirm_reset: false` conserva i dati presenti. Se tabella/collezione è popolata, il caricamento si ferma: nessun append o reset implicito. Un reset richiede il flag booleano `true` e la verifica dei nomi PostgreSQL schema/table o Mongo database/collection mostrati sopra; riportare il flag a `false` dopo l'uso. Non si eliminano interi database o volumi.
PostgreSQL usa executemany e commit atomico per batch; Mongo usa insert_many ordinato con w=1 e journal, senza transazione multi-documento o retry. Un batch Mongo fallito può essere parziale: contatori e incertezza dell'ack sono registrati, senza pulizia automatica. Anche DROP/CREATE Mongo non sono una transazione atomica.
CockroachDB single-node usa INT8/INT4/STRING e transazioni SERIALIZABLE per batch. Ripete soltanto l'intera transazione abortita con SQLSTATE 40001, con limite e backoff registrati. Errori di rete/commit ambiguo interrompono il run senza retry. Retry e attese sono inclusi nei tempi; la modalità insecure è solo locale e non adatta alla produzione.
`duration_ms` va dalla prima chiamata di scrittura all'ultimo commit/ack e comprende la lettura dei batch successivi. `driver_write_ms` somma le chiamate all'adapter: per Mongo comprende costruzione BSON, per Cockroach anche retry/backoff. Checksum, preprocessing, prima lettura CSV, DDL e indici sono fuori dal timer. I timeout 100k restano 120s per statement e 10s per lock: eventuali audit su altre dimensioni vanno gestiti dal runner, senza aumenti automatici dei timeout.

In [ ]:
if ABILITA_INGESTIONE is True:
    with ResultRun(config.results_dir, config.manifest(), reference, stage='ingestion') as run:
        adapter = create_adapter(config)
        try:
            adapter.connect()
            run.set_database_version(adapter.version())
            if config.confirm_reset:
                adapter.reset_dataset(confirm_reset=True)
            else:
                adapter.create_schema()
            try:
                ingestion = ingest_csv(adapter, csv_path, config.dataset_size, artifact['sha256'], batch_size=config.batch_size)
            except IngestionError as error:
                run.save_json('ingestion_failed.json', error.progress)
                raise
            run.save_json('ingestion.json', ingestion)
            run.record(event='ingestion', index_state='primary_only', result_digest=result_digest(ingestion['rows_committed']), **ingestion)
            correctness = verify_database(adapter, reference, 'primary_only')
            run.save_json('correctness_primary_only.json', correctness)
            if config.database in ('postgres', 'cockroach'):
                adapter.analyze()
            if config.database == 'postgres':
                run.save_json('database_snapshot.json', capture_database_snapshot(adapter, reference, 'primary_only'))
            elif config.database == 'mongo':
                run.save_json('postgres_comparison.json', compare_postgres_snapshot(adapter, reference, config.postgres_snapshot, 'primary_only'))
            else:
                run.save_json('reference_comparison.json', compare_reference_snapshots(adapter, reference, config.postgres_snapshot, config.mongo_snapshot, 'primary_only'))
        finally:
            adapter.close()
    print('Run:', run.run_id, '| directory:', run.directory)
    print(json.dumps(ingestion, indent=2))
    print('Correttezza:', correctness['status'])
else:
    print('Operazione disabilitata: ABILITA_INGESTIONE=False.')


## Persistenza e scelta del percorso
Il run conserva manifest, reference, raw JSONL, summary CSV e diagnostica; per Mongo include il confronto con lo snapshot PostgreSQL e l'audit BSON nella verifica. Solo `status=completed` identifica una fase completa. Le credenziali restano nell'ambiente del container, mai nel manifest. Il DB conserva dati e indici nel volume anche arrestando il container.
Ora aprire `03_query_benchmarks.ipynb` senza ricaricare il dataset. Non rieseguire questo notebook sulla tabella popolata senza una scelta esplicita. Per campagne sistematiche usare invece `04_full_runner.ipynb`; per la presentazione usare `05_demo.ipynb`. Non alternare percorso manuale e runner sulla stessa destinazione.